In [ ]:
#Install  Libraries 
!pip install datasets torchvision scikit-learn


In [ ]:
#  Imports 
import os
import random
import numpy as np
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as T
import torchvision.models as models

from datasets import load_dataset

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
# Device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


Using device: cpu


In [ ]:
#  Load VQA-RAD Dataset
dataset = load_dataset("flaviagiammarino/vqa-rad")

print(dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 1793
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 451
    })
})


In [ ]:
#  Image Transform 
image_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
#  Simple Tokenizer
class SimpleTokenizer:
    def __init__(self, texts, max_len=32):
        self.max_len = max_len
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}

        idx = 2
        for text in texts:
            for word in text.lower().split():
                if word not in self.word2idx:
                    self.word2idx[word] = idx
                    idx += 1

    def encode(self, text):
        tokens = [
            self.word2idx.get(w, self.word2idx["<UNK>"])
            for w in text.lower().split()
        ]

        if len(tokens) < self.max_len:
            tokens += [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]

        return torch.tensor(tokens)


In [ ]:
#  Answer Vocabulary 
train_answers = [ex["answer"] for ex in dataset["train"]]

answer2idx = {"<UNK>": 0}
idx = 1
for ans in set(train_answers):
    answer2idx[ans] = idx
    idx += 1

idx2answer = {v: k for k, v in answer2idx.items()}

print("Number of answer classes (including <UNK>):", len(answer2idx))


Number of answer classes (including <UNK>): 433


In [ ]:
#VQA Dataset Class 
class VQARadDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, answer2idx):
        self.data = hf_dataset
        self.tokenizer = tokenizer
        self.answer2idx = answer2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Image
        image = item["image"].convert("RGB")
        image = image_transform(image)

        # Question
        question = self.tokenizer.encode(item["question"])

        # Answer (SAFE mapping)
        answer_text = item["answer"]
        answer_id = self.answer2idx.get(answer_text, self.answer2idx["<UNK>"])
        answer = torch.tensor(answer_id)

        return image, question, answer


In [ ]:
#Tokenizer (fit on TRAIN questions only)
train_questions = [ex["question"] for ex in dataset["train"]]
tokenizer = SimpleTokenizer(train_questions, max_len=32)

#  Full Training Dataset
full_train_dataset = VQARadDataset(
    dataset["train"],
    tokenizer,
    answer2idx
)

train_size = int(0.85 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

#Test Dataset 
test_dataset = VQARadDataset(
    dataset["test"],
    tokenizer,
    answer2idx
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))


Train: 1524
Val: 269
Test: 451


In [ ]:
# DataLoaders 
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)


In [ ]:
#  CNN-BiLSTM Model 
class CNN_BiLSTM(nn.Module):
    def __init__(self, vocab_size, num_answers, embed_dim=300, hidden_dim=256):
        super().__init__()

        # CNN (ResNet-50)
        resnet = models.resnet50(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        self.img_fc = nn.Linear(2048, hidden_dim)

        # Text Encoder
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_answers)
        )

    def forward(self, image, question):
        # Image
        img_feat = self.cnn(image).squeeze()
        img_feat = self.img_fc(img_feat)

        # Text
        emb = self.embedding(question)
        _, (h, _) = self.lstm(emb)
        txt_feat = torch.cat([h[0], h[1]], dim=1)

        # Fusion
        fused = torch.cat([img_feat, txt_feat], dim=1)
        return self.classifier(fused)


In [ ]:
# Training
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, questions, answers in loader:
        images = images.to(device)
        questions = questions.to(device)
        answers = answers.to(device)

        optimizer.zero_grad()
        logits = model(images, questions)
        loss = criterion(logits, answers)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
# Evaluation 
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, questions, answers in loader:
            images = images.to(device)
            questions = questions.to(device)

            logits = model(images, questions)
            preds = logits.argmax(dim=1).cpu().numpy()

            y_pred.extend(preds)
            y_true.extend(answers.numpy())

    # Exact Match Accuracy
    exact_match = accuracy_score(y_true, y_pred)

    # Yes/No metrics
    yesno_ids = [
        idx for idx, ans in idx2answer.items()
        if ans.lower() in ["yes", "no"]
    ]

    yn_true, yn_pred = [], []
    for t, p in zip(y_true, y_pred):
        if t in yesno_ids:
            yn_true.append(t)
            yn_pred.append(p)

    if len(yn_true) > 0:
        p, r, f, _ = precision_recall_fscore_support(
            yn_true, yn_pred, average="macro", zero_division=0
        )
        cm = confusion_matrix(yn_true, yn_pred)
    else:
        p = r = f = cm = None

    return exact_match, p, r, f, cm


In [ ]:
# Initialize Model 
model = CNN_BiLSTM(
    vocab_size=len(tokenizer.word2idx),
    num_answers=len(answer2idx)
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

#  Train 
epochs = 10

for epoch in range(epochs):
    loss = train_epoch(model, train_loader, optimizer, criterion)
    acc, p, r, f, cm = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Loss: {loss:.4f}")
    print(f"Exact Match Accuracy: {acc:.4f}")
    if p is not None:
        print(f"Yes/No Precision: {p:.4f}, Recall: {r:.4f}, F1: {f:.4f}")
    print("-" * 40)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/10
Loss: 4.3635
Exact Match Accuracy: 0.3346
Yes/No Precision: 0.6391, Recall: 0.6385, F1: 0.6380
----------------------------------------
Epoch 2/10
Loss: 3.4967
Exact Match Accuracy: 0.3197
Yes/No Precision: 0.6237, Recall: 0.6111, F1: 0.6002
----------------------------------------
Epoch 3/10
Loss: 3.2536
Exact Match Accuracy: 0.3866
Yes/No Precision: 0.7126, Recall: 0.7088, F1: 0.7077
----------------------------------------
Epoch 4/10
Loss: 3.1461
Exact Match Accuracy: 0.4089
Yes/No Precision: 0.7208, Recall: 0.7168, F1: 0.7152
----------------------------------------
Epoch 5/10
Loss: 2.9204
Exact Match Accuracy: 0.4052
Yes/No Precision: 0.6983, Recall: 0.6955, F1: 0.6941
----------------------------------------
Epoch 6/10
Loss: 2.8058
Exact Match Accuracy: 0.3941
Yes/No Precision: 0.6968, Recall: 0.6749, F1: 0.6648
----------------------------------------
Epoch 7/10
Loss: 2.6518
Exact Match Accuracy: 0.4461
Yes/No Precision: 0.7455, Recall: 0.7449, F1: 0.7446
------------

In [ ]:
#Test Set Evaluation 
test_acc, test_p, test_r, test_f, test_cm = evaluate(model, test_loader)

print("TEST RESULTS")
print("Exact Match Accuracy:", test_acc)

if test_p is not None:
    print("Yes/No Precision:", test_p)
    print("Yes/No Recall:", test_r)
    print("Yes/No F1:", test_f)
    print("Confusion Matrix:\n", test_cm)


TEST RESULTS
Exact Match Accuracy: 0.35920177383592017
Yes/No Precision: 0.4311135571454779
Yes/No Recall: 0.42247568072724184
Yes/No F1: 0.4153623188405797
Confusion Matrix:
 [[ 0  0  0]
 [ 0 66 67]
 [ 1 26 91]]
